# Choice Modeling with Choice-Based Conjoint

## Learning objectives
By the end of this notebook, you will be able to:

1. explain how conjoint analysis decomposes product utility into attribute-level part-worths;
2. design and validate a choice-based conjoint (CBC) experiment;
3. estimate a grouped conditional logit model;
4. translate coefficients into attribute importance and willingness to pay; and
5. communicate the assumptions and limitations of the analysis.

> **Updated workflow:** This notebook uses a reproducible synthetic experiment, so it runs without external data files. Replace the simulated choices with real survey exports when applying the workflow.

## 1. From product features to choices

Conjoint analysis estimates how people trade off product attributes. Instead of asking whether price or screen quality is important in the abstract, a CBC survey presents realistic alternatives and asks respondents to choose one.

For a television study, a choice task might vary:

- **brand**;
- **screen size**;
- **display technology**;
- **energy rating**; and
- **price**.

Each observed choice reveals information about the respondent's relative preferences. The resulting coefficients are called **part-worth utilities**. They are interval-scaled: differences are meaningful, but their absolute zero is not.

### CBC data structure

The data use **long format**: one row per alternative shown in a task. Exactly one alternative must be marked as chosen within each choice set.

| respondent | task | choice_set | alternative | brand | screen | display | energy | price | chosen |
|---:|---:|---:|---:|---|---|---|---|---:|---:|
| 1 | 1 | 1 | 1 | Aster | 55 | OLED | A | 1200 | 0 |
| 1 | 1 | 1 | 2 | Beacon | 65 | OLED | A | 1500 | 1 |
| 1 | 1 | 1 | 3 | Cobalt | 75 | Mini-LED | B | 900 | 0 |

The `choice_set` identifier is essential: alternatives compete only with the other alternatives shown in the same task.

## 2. Utility model

For respondent $r$, task $t$, and alternative $j$:

$$U_{rtj} = \mathbf{x}_{rtj}^{\mathsf{T}}\boldsymbol{\beta} + \varepsilon_{rtj}$$

The observed choice is the alternative with the highest latent utility. Under independent type-I extreme-value errors, the conditional logit choice probability is:

$$P(y_{rt}=j)=\frac{\exp(\mathbf{x}_{rtj}^{\mathsf{T}}\boldsymbol{\beta})}{\sum_{k \in C_{rt}}\exp(\mathbf{x}_{rtk}^{\mathsf{T}}\boldsymbol{\beta})}$$

Here, $C_{rt}$ is the set of alternatives shown in that task. Alternative-specific constants or a **none** option can be added when the survey design supports them.

### Experimental-design checklist

Before collecting responses:

1. choose attributes and levels that are understandable, actionable, and mutually exclusive;
2. remove impossible product combinations;
3. randomize task and alternative order;
4. balance level frequencies and limit correlations between attributes;
5. include holdout tasks for validation; and
6. pilot the survey to check comprehension and task burden.

CBC estimates **stated preferences**. Results can be biased by hypothetical choices, poor attribute framing, ignored alternatives, or an unrepresentative sample.

## 3. Reproducible teaching experiment

We simulate 250 respondents, each completing eight three-alternative tasks. The values below are **data-generating assumptions**, not research findings. A random seed makes the example reproducible.

The model uses reference coding for categorical attributes and treats price as linear per $100. In production, test whether a linear price effect is credible; categorical price levels or splines may be safer when the relationship is nonlinear.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from IPython.display import Markdown, display
from statsmodels.discrete.conditional_models import ConditionalLogit

pd.set_option("display.max_columns", 20)
plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
levels = {
    "brand": ["Aster", "Beacon", "Cobalt"],
    "screen_size": ["55", "65", "75"],
    "display": ["OLED", "Mini-LED"],
    "energy_rating": ["A", "B"],
    "price": [900, 1200, 1500],
}

profiles = pd.DataFrame(
    product(*levels.values()),
    columns=levels.keys(),
)
profiles.index.name = "profile_id"
print(f"Full-factorial candidate set: {len(profiles):,} profiles")

In [ ]:
profiles.sample(8, random_state=RANDOM_SEED).sort_index()

In [ ]:
# Utilities used only to generate the teaching data.
true_part_worths = {
    "brand": {"Aster": 0.00, "Beacon": 0.35, "Cobalt": -0.20},
    "screen_size": {"55": 0.00, "65": 0.30, "75": 0.50},
    "display": {"OLED": 0.00, "Mini-LED": -0.25},
    "energy_rating": {"A": 0.00, "B": -0.20},
}
true_price_per_100 = -0.18

rows = []
n_respondents, n_tasks, n_alternatives = 250, 8, 3

for respondent in range(1, n_respondents + 1):
    respondent_price = true_price_per_100 + rng.normal(0, 0.025)
    for task in range(1, n_tasks + 1):
        sampled = profiles.iloc[
            rng.choice(len(profiles), size=n_alternatives, replace=False)
        ].copy()
        deterministic = (
            sampled["brand"].map(true_part_worths["brand"])
            + sampled["screen_size"].map(true_part_worths["screen_size"])
            + sampled["display"].map(true_part_worths["display"])
            + sampled["energy_rating"].map(true_part_worths["energy_rating"])
            + respondent_price * sampled["price"] / 100
        )
        chosen_position = np.argmax(deterministic.to_numpy() + rng.gumbel(size=n_alternatives))
        choice_set = f"R{respondent:03d}_T{task:02d}"
        for alternative, (_, profile) in enumerate(sampled.iterrows(), start=1):
            rows.append({
                "respondent": respondent,
                "task": task,
                "choice_set": choice_set,
                "alternative": alternative,
                **profile.to_dict(),
                "chosen": int(alternative - 1 == chosen_position),
            })

choices = pd.DataFrame(rows)
choices.head(6)

In [ ]:
display(Markdown(r"""## 4. Validate the choice data

A valid CBC file should have one chosen alternative per choice set, no duplicated alternatives within a set, and no missing model inputs. These checks should run before estimation."""))

In [ ]:
set_sizes = choices.groupby("choice_set").size()
chosen_per_set = choices.groupby("choice_set")["chosen"].sum()

assert set_sizes.eq(n_alternatives).all(), "Each task must contain the expected alternatives."
assert chosen_per_set.eq(1).all(), "Each task must contain exactly one choice."
assert not choices.isna().any().any(), "Model inputs contain missing values."
assert not choices.duplicated(["choice_set", "alternative"]).any()

pd.Series({
    "respondents": choices["respondent"].nunique(),
    "choice_sets": choices["choice_set"].nunique(),
    "rows": len(choices),
    "chosen_rows": int(choices["chosen"].sum()),
})

In [ ]:
# Level balance is a basic design diagnostic.
level_balance = {
    column: choices[column].value_counts(normalize=True).sort_index()
    for column in ["brand", "screen_size", "display", "energy_rating", "price"]
}
pd.concat(level_balance, axis=1).fillna(0).round(3)

In [ ]:
display(Markdown(r"""## 5. Encode attributes and estimate the model

Reference levels are `Aster`, `55`, `OLED`, and energy rating `A`. Their part-worths are fixed to zero for identification. Price is expressed in hundreds of dollars so its coefficient is numerically readable.

A grouped conditional logit is appropriate because every outcome is a choice among alternatives in the same task. Ordinary least squares is not appropriate for this binary, grouped outcome."""))

In [ ]:
categorical = ["brand", "screen_size", "display", "energy_rating"]
X = pd.get_dummies(
    choices[categorical],
    columns=categorical,
    drop_first=True,
    dtype=float,
)
X["price_per_100"] = choices["price"] / 100
X.head()

In [ ]:
model = ConditionalLogit(
    endog=choices["chosen"],
    exog=X,
    groups=choices["choice_set"],
)
result = model.fit(method="bfgs", maxiter=300, disp=False)
print(result.summary())

In [ ]:
display(Markdown(r"""### Reading the coefficients

- A positive coefficient raises utility relative to its reference level.
- A negative price coefficient means higher prices reduce choice probability.
- A 95% confidence interval that excludes zero is evidence against a zero average effect under the model assumptions.
- Coefficient magnitude is comparable **within this model**, but raw coefficients are not percentages or probabilities.

Because the example contains repeated choices per respondent, production inference should account for respondent-level dependence. Depending on the software and study design, use respondent-clustered uncertainty, a mixed logit, or a hierarchical Bayes model."""))

In [ ]:
coef_table = pd.DataFrame({
    "coefficient": result.params,
    "std_error": result.bse,
    "p_value": result.pvalues,
})
coef_table["ci_low"] = result.conf_int()[0]
coef_table["ci_high"] = result.conf_int()[1]
coef_table.round(3)

In [ ]:
display(Markdown(r"""## 6. Part-worth utilities

Add the omitted reference levels back at zero to create a complete level table. Price is handled separately because it is modeled as a continuous attribute."""))

In [ ]:
reference_levels = {
    "brand": "Aster",
    "screen_size": "55",
    "display": "OLED",
    "energy_rating": "A",
}

part_worth_rows = []
for attribute, reference in reference_levels.items():
    part_worth_rows.append({"attribute": attribute, "level": reference, "utility": 0.0})
    prefix = f"{attribute}_"
    for parameter, value in result.params.items():
        if parameter.startswith(prefix):
            part_worth_rows.append({
                "attribute": attribute,
                "level": parameter.removeprefix(prefix),
                "utility": value,
            })

part_worths = pd.DataFrame(part_worth_rows)
part_worths.sort_values(["attribute", "utility"], ascending=[True, False]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = part_worths.sort_values("utility")
labels = plot_data["attribute"] + ": " + plot_data["level"].astype(str)
colors = np.where(plot_data["utility"] >= 0, "#2a9d8f", "#e76f51")
ax.barh(labels, plot_data["utility"], color=colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set(title="Estimated part-worth utilities", xlabel="Utility relative to reference level")
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(r"""## 7. Relative attribute importance

A common descriptive measure is the utility range of each attribute over the tested levels:

$$I_a = \max_l(\hat{u}_{al}) - \min_l(\hat{u}_{al})$$

Normalized importance divides each range by the sum of all ranges. For continuous price, use the utility change across the **tested price range**. Importance therefore depends on the selected levels and should not be generalized to a wider range."""))

In [ ]:
ranges = part_worths.groupby("attribute")["utility"].agg(lambda s: s.max() - s.min())
price_range_utility = abs(
    result.params["price_per_100"] * (max(levels["price"]) - min(levels["price"])) / 100
)
ranges.loc["price"] = price_range_utility

importance = (
    (100 * ranges / ranges.sum())
    .sort_values(ascending=False)
    .rename("relative_importance_pct")
    .to_frame()
)
importance.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
importance.sort_values("relative_importance_pct").plot.barh(
    ax=ax, legend=False, color="#457b9d"
)
ax.set(title="Relative attribute importance", xlabel="Share of total utility range (%)", ylabel="")
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(r"""## 8. Willingness to pay

When price has a negative linear coefficient, the marginal willingness to pay (WTP) for a level relative to its reference is:

$$WTP_l = -\frac{\beta_l}{\beta_{price}}$$

The estimate inherits all model assumptions and can become unstable when the price coefficient is near zero. Use confidence intervals or simulation-based uncertainty in research reports."""))

In [ ]:
price_beta_per_dollar = result.params["price_per_100"] / 100
wtp = part_worths.query("utility != 0").copy()
wtp["wtp_vs_reference_usd"] = -wtp["utility"] / price_beta_per_dollar
wtp[["attribute", "level", "wtp_vs_reference_usd"]].round(0)

In [ ]:
display(Markdown(r"""## 9. Descriptive fit check

The hit rate below asks whether the alternative with the largest fitted utility was actually chosen in each task. It is an **in-sample diagnostic**, not an estimate of future accuracy. Use respondent-level train/test splits or dedicated holdout tasks for honest validation."""))

In [ ]:
choices_with_fit = choices.copy()
choices_with_fit["fitted_utility"] = X.to_numpy() @ result.params.to_numpy()
choices_with_fit["predicted_choice"] = (
    choices_with_fit.groupby("choice_set")["fitted_utility"]
    .transform(lambda s: s.eq(s.max()))
    .astype(int)
)

hit_rate = (
    choices_with_fit.loc[choices_with_fit["predicted_choice"].eq(1), "chosen"].mean()
)
print(f"In-sample hit rate: {hit_rate:.1%}")
print(f"Random-choice benchmark: {1 / n_alternatives:.1%}")

In [ ]:
display(Markdown(r"""## 10. Scenario simulation

A share-of-preference simulator converts utilities into logit probabilities for a proposed competitive set. These are model-based preference shares, not guaranteed market shares: awareness, distribution, stock, promotion, and outside options also influence real purchases."""))

In [ ]:
def preference_shares(products: pd.DataFrame) -> pd.DataFrame:
    """Return conditional-logit shares for one competitive scenario."""
    encoded = pd.get_dummies(products[categorical], columns=categorical, drop_first=True, dtype=float)
    encoded = encoded.reindex(columns=X.columns.drop("price_per_100"), fill_value=0.0)
    encoded["price_per_100"] = products["price"].to_numpy() / 100
    utility = encoded.to_numpy() @ result.params.to_numpy()
    stable_exp = np.exp(utility - utility.max())
    output = products.copy()
    output["predicted_share"] = stable_exp / stable_exp.sum()
    return output.sort_values("predicted_share", ascending=False)

In [ ]:
scenario = pd.DataFrame([
    {"brand": "Aster", "screen_size": "55", "display": "OLED", "energy_rating": "A", "price": 900},
    {"brand": "Beacon", "screen_size": "65", "display": "OLED", "energy_rating": "A", "price": 1200},
    {"brand": "Cobalt", "screen_size": "75", "display": "Mini-LED", "energy_rating": "B", "price": 1500},
])
scenario_results = preference_shares(scenario)
scenario_results.style.format({"price": "${:,.0f}", "predicted_share": "{:.1%}"})

In [ ]:
display(Markdown(r"""## 11. What this basic model does not capture

- **Preference heterogeneity:** average effects can hide segments with opposing preferences.
- **Repeated observations:** choices from one respondent are correlated.
- **IIA:** standard logit imposes restrictive substitution patterns.
- **Nonlinear price response:** one coefficient assumes constant marginal utility of money.
- **Survey uncertainty:** attribute non-attendance, fatigue, and hypothetical bias can distort responses.
- **Causal interpretation:** conjoint randomization supports causal attribute effects only when the experimental design and survey implementation are valid.

Mixed logit and hierarchical Bayes models are common next steps when respondent-level heterogeneity matters."""))

In [ ]:
display(Markdown(r"""## 12. Practice

1. Increase Beacon's price in the scenario and plot its predicted share.
2. Treat price as categorical and compare the implied pattern with the linear model.
3. Reserve the final two tasks per respondent as holdouts and calculate a true holdout hit rate.
4. Add a **none** alternative and discuss how its utility should be represented.
5. Replace the synthetic data with a survey export that follows the validated long format."""))

In [ ]:
# Exercise starter: evaluate Beacon across several prices.
price_grid = pd.DataFrame([
    {"brand": "Beacon", "screen_size": "65", "display": "OLED", "energy_rating": "A", "price": price}
    for price in range(900, 1501, 100)
])
price_grid

In [ ]:
display(Markdown(r"""## Key takeaway

Choice-based conjoint turns repeated trade-off decisions into estimates of relative utility. A defensible analysis requires more than fitting a model: the survey design, choice-set identifiers, reference coding, validation strategy, uncertainty, and behavioral assumptions all determine whether the results are useful."""))